# Lost in the Middle — Qwen 2.5 7B | Closed-Book Baseline
**Model:** qwen2.5:7b-instruct | All 2,655 questions | **Closed-Book Baseline (No Documents)**  
**GPU:** 2x T4 | Estimated time: ~2–3 hours  

This notebook runs the closed-book baseline evaluation on Qwen 2.5 7B. The model is prompted only with the question and instructions to answer concisely, completely ignoring context documents. This sets the lower-bound baseline for the study.

**Evaluation follows the original paper exactly:**
- Response truncated at first `\n` before matching
- All valid answer synonyms checked (`data["answers"]`)
- `normalize_answer`: lowercase → remove articles → remove punctuation → fix whitespace
- temperature=0 (greedy decoding), full 2,655 questions
- Concise answer prompt (keeps inference fast: ~2–3s per question)

In [ ]:
!pip install ollama

In [ ]:
import subprocess
import time

print("Installing dependencies...")
subprocess.run("apt-get update && apt-get install -y zstd", shell=True, check=True)
subprocess.run("pip install ollama==0.6.2", shell=True, check=True)

print("Installing Ollama...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

print("Starting server...")
subprocess.Popen("ollama serve", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(10)

print("Pulling model...")
subprocess.run("ollama pull qwen2.5:7b-instruct", shell=True, check=True)
print("Ollama ready!")

In [ ]:
import os
import json
import gzip
import string
import re
import time
import ollama

# =====================================================
# CONFIG
# =====================================================
MODEL_NAME = "qwen2.5:7b-instruct"
NUM_CTX = 4096
DATASET_PATH = "/kaggle/input/datasets/arinjaysarkar/qa-questions"
OUTPUT_DIR = "/kaggle/working"
NUM_QUESTIONS_TO_TEST = 2655

# =====================================================
# NORMALIZATION (paper-exact)
# =====================================================
def normalize_answer(s: str) -> str:
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)
    def white_space_fix(text):
        return " ".join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

# =====================================================
# LOAD / RESUME SAVED RESPONSES
# =====================================================
responses_file = os.path.join(OUTPUT_DIR, "all_responses_qwen25_baseline.json")
results_file   = os.path.join(OUTPUT_DIR, "results_qwen25_baseline.json")

if os.path.exists(responses_file):
    with open(responses_file, "r") as f:
        all_responses = json.load(f)
    print(f"Loaded {len(all_responses)} saved responses — resuming!")
else:
    all_responses = []
    print("No saved responses found — starting fresh.")

start_q = len(all_responses)
correct_matches = sum(1 for r in all_responses if r["match"])
print(f"Starting from Q{start_q} | Correct so far: {correct_matches}")

# =====================================================
# DATASET FILE (using 30-doc gold-at-0 jsonl just to get questions)
# =====================================================
file_path = os.path.join(
    DATASET_PATH, "qa_data", "30_total_documents",
    "nq-open-30_total_documents_gold_at_0.jsonl"
)
print(f"Loading: {file_path}")

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

total_questions = len(lines)
run_start = time.time()

print(f"Total questions: {total_questions}")
print("=" * 60)

# =====================================================
# EVALUATION LOOP
# =====================================================
for q_idx in range(start_q, total_questions):

    data = json.loads(lines[q_idx])
    target_query   = data["question"]
    target_answers = data["answers"]

    # Closed-book baseline prompt (no documents)
    prompt = (
        "Write a high-quality answer for the given question.\n\n"
        f"Question: {target_query}\n"
        "Answer: (Keep the answer as concise as possible, preferably one or two words. "
        "Do not write full sentences.)"
    )

    try:
        q_start = time.time()
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            options={"num_ctx": NUM_CTX, "temperature": 0.0}
        )
        q_time = time.time() - q_start

        raw_output = response["message"]["content"]
        output = raw_output.split("\n")[0].strip()           # first-line truncation
        normalized_prediction = normalize_answer(output)

        success = any(
            normalize_answer(ans) in normalized_prediction
            for ans in target_answers
        )
        if success:
            correct_matches += 1

        all_responses.append({
            "q_idx": q_idx,
            "question": target_query,
            "target_answers": target_answers,
            "raw_output": raw_output.strip(),
            "truncated_output": output,
            "match": success,
            "time_sec": round(q_time, 2)
        })

        # Save every 50 questions
        if (q_idx + 1) % 50 == 0 or q_idx == total_questions - 1:
            with open(responses_file, "w") as f:
                json.dump(all_responses, f)
            acc_so_far = (correct_matches / (q_idx + 1)) * 100
            elapsed = time.time() - run_start
            eta = (elapsed / (q_idx - start_q + 1)) * (total_questions - q_idx - 1) if (q_idx - start_q + 1) > 0 else 0
            print(
                f"[Q{q_idx+1}/{total_questions}] ({q_time:.1f}s) "
                f"Acc: {acc_so_far:.1f}% | "
                f"ETA: {eta/60:.0f}min | "
                f"{target_answers[0]} -> {output[:40]} | {success}"
            )

    except Exception as e:
        print(f"Error at Q{q_idx+1}: {e}")
        all_responses.append({
            "q_idx": q_idx, "question": target_query, "target_answers": target_answers,
            "raw_output": f"ERROR: {str(e)}",
            "truncated_output": "", "match": False, "time_sec": 0
        })

# =====================================================
# FINAL RESULT
# =====================================================
final_correct = sum(1 for r in all_responses if r["match"])
accuracy = (final_correct / total_questions) * 100
total_time = time.time() - run_start

print(f"\n{'='*60}")
print(f"BASELINE EVAL DONE: {accuracy:.2f}% ({final_correct}/{total_questions}) [{total_time/60:.1f}min]")
print(f"{'='*60}")

results_data = [{
    "Context Size (Documents)": "Baseline",
    "Position of Answer (Gold Index)": -1,
    "Accuracy (%)": round(accuracy, 5),
    "Correct": final_correct,
    "Total": total_questions
}]

with open(results_file, "w") as f:
    json.dump(results_data, f, indent=2)

print(f"\nComplete baseline results saved to: {results_file}")
print(json.dumps(results_data, indent=2))